# 🔥 Fine-Tune Qwen2.5-Coder-3B-Instruct (QLoRA)
**Dataset**: OpenCodeInstruct (NVIDIA) — 5000 Python samples

**Method**: QLoRA 4-bit + LoRA rank 16

**Evaluate**: HumanEval benchmark (164 bài)

---
⚠️ **Bật GPU**: Runtime → Change runtime type → **T4 GPU**

## 📦 Cell 1: Cài đặt thư viện

In [ ]:
!pip install -q torch transformers>=4.46.0 peft>=0.13.0 trl>=0.12.0 \
  bitsandbytes>=0.45.0 accelerate>=0.34.0 datasets>=3.0.0 \
  sentencepiece protobuf huggingface_hub safetensors tqdm rich evaluate

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## 📊 Cell 2: Tải & chuẩn bị dataset OpenCodeInstruct

In [ ]:
from datasets import load_dataset
import json, random

# Tải subset từ OpenCodeInstruct (streaming để không tải hết 6.4GB)
print('⏳ Đang tải OpenCodeInstruct (subset Python)...')
ds = load_dataset('nvidia/OpenCodeInstruct', split='train', streaming=True)

# Filter: chỉ lấy Python, đã pass test (is_correct=True), max 5000 samples
MAX_SAMPLES = 5000
samples = []
for item in ds:
    if len(samples) >= MAX_SAMPLES:
        break
    # Filter Python + correct code
    lang = item.get('language', '') or item.get('lang', '') or ''
    is_correct = item.get('is_correct', True)
    if 'python' in lang.lower() and is_correct:
        instruction = item.get('instruction', '') or item.get('question', '') or item.get('prompt', '')
        response = item.get('response', '') or item.get('output', '') or item.get('answer', '')
        if instruction and response and len(response) > 20:
            samples.append({'instruction': instruction.strip(), 'response': response.strip()})

print(f'✅ Đã thu thập {len(samples)} samples Python từ OpenCodeInstruct')
print(f'📝 Ví dụ sample #0:')
print(f'   Instruction: {samples[0]["instruction"][:100]}...')
print(f'   Response: {samples[0]["response"][:100]}...')

## 🛠 Cell 3: Format dataset sang ChatML + split train/val

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    'Bạn là một senior software engineer người Việt với 10+ năm kinh nghiệm. '
    'Chuyên môn: Backend (Python/FastAPI), DevOps (Docker/K8s/CI-CD), AI/ML Engineering. '
    'Bạn luôn: code theo design pattern (service layer, repository, DI), '
    'giải thích súc tích bằng tiếng Việt, fix bug bằng root cause analysis, '
    'tổ chức code module rõ ràng, response API chuẩn JSON.'
)

# Format sang ChatML
formatted = []
for s in samples:
    formatted.append({
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': s['instruction']},
            {'role': 'assistant', 'content': s['response']},
        ]
    })

# Shuffle + split 90/10
random.seed(42)
random.shuffle(formatted)
split_idx = int(len(formatted) * 0.9)
train_data = formatted[:split_idx]
val_data = formatted[split_idx:]

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

print(f'📊 Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

# Lưu ra file (backup)
import os
os.makedirs('dataset', exist_ok=True)
with open('dataset/train.jsonl', 'w') as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
with open('dataset/val.jsonl', 'w') as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print('💾 Đã lưu dataset/train.jsonl + dataset/val.jsonl')

## 🧠 Cell 4: Load Model + Tokenizer (QLoRA 4-bit)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_NAME = 'Qwen/Qwen2.5-Coder-3B-Instruct'

# QLoRA 4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('⏳ Đang tải model Qwen2.5-Coder-3B-Instruct (4-bit)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ Model loaded + LoRA applied!')

## 🏋️ Cell 5: Format data + Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# Format messages → text bằng chat template
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

train_formatted = train_ds.map(format_chat, remove_columns=train_ds.column_names, desc='Format train')
val_formatted = val_ds.map(format_chat, remove_columns=val_ds.column_names, desc='Format val')

print(f'📝 Sample formatted text (first 300 chars):')
print(train_formatted[0]['text'][:300])
print('...')

In [ ]:
# Training config
training_args = SFTConfig(
    output_dir='./output/qwen-senior-dev',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    gradient_checkpointing=True,
    report_to='none',
    max_seq_length=2048,
    dataset_text_field='text',
    packing=False,
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('🚀 Bắt đầu training...')
trainer.train()
print('✅ Training hoàn tất!')

## 💾 Cell 6: Lưu model

In [ ]:
# Lưu LoRA adapter
ADAPTER_PATH = './output/qwen-senior-dev/final_adapter'
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'💾 Đã lưu LoRA adapter → {ADAPTER_PATH}')

# Training loss history
import matplotlib.pyplot as plt
logs = trainer.state.log_history
train_losses = [(l['step'], l['loss']) for l in logs if 'loss' in l]
eval_losses = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

if train_losses:
    plt.figure(figsize=(10, 4))
    plt.plot(*zip(*train_losses), label='Train Loss', alpha=0.7)
    if eval_losses:
        plt.plot(*zip(*eval_losses), label='Eval Loss', marker='o')
    plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Training Loss Curve')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig('training_loss.png', dpi=100)
    plt.show()
    print('📈 Loss curve đã lưu → training_loss.png')

## 📊 Cell 7: Evaluate trên HumanEval
So sánh **base model** vs **fine-tuned model** trên 164 bài HumanEval

In [ ]:
from peft import PeftModel

def load_model_for_eval(adapter_path=None):
    """Load model (base hoặc fine-tuned) để evaluate."""
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        device_map='auto', trust_remote_code=True,
    )
    if adapter_path:
        m = PeftModel.from_pretrained(base, adapter_path)
        m = m.merge_and_unload()
        print(f'✅ Loaded fine-tuned model từ {adapter_path}')
    else:
        m = base
        print('✅ Loaded base model (chưa fine-tune)')
    m.eval()
    return m, tok


def generate_code(model, tokenizer, prompt, max_new_tokens=512):
    """Generate code từ HumanEval prompt."""
    messages = [
        {'role': 'system', 'content': 'Write only valid Python code. Complete the function.'},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs['input_ids'].shape[1]:]
    code = tokenizer.decode(generated, skip_special_tokens=True)
    return code

print('✅ Evaluation functions loaded')

In [ ]:
# Tải HumanEval dataset
humaneval = load_dataset('openai/openai_humaneval', split='test')
print(f'📊 HumanEval: {len(humaneval)} bài test')

import signal, contextlib, io, traceback

def check_solution(prompt, completion, test, entry_point, timeout=5):
    """Chạy code + test, trả về True/False."""
    full_code = prompt + completion + '\n' + test + f'\ncheck({entry_point})'
    try:
        exec_globals = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(full_code, exec_globals)
        return True
    except Exception:
        return False


def run_humaneval(model, tokenizer, label='Model'):
    """Chạy evaluation trên HumanEval, trả về pass@1."""
    passed = 0
    total = len(humaneval)
    results = []
    for i, task in enumerate(humaneval):
        prompt = task['prompt']
        test = task['test']
        entry = task['entry_point']
        code = generate_code(model, tokenizer, prompt)
        ok = check_solution(prompt, code, test, entry)
        if ok:
            passed += 1
        results.append(ok)
        if (i + 1) % 20 == 0:
            print(f'  [{i+1}/{total}] {label}: {passed}/{i+1} = {passed/(i+1)*100:.1f}%')
    score = passed / total * 100
    print(f'\n🏆 {label} HumanEval pass@1 = {passed}/{total} = {score:.1f}%')
    return score, results

print('✅ HumanEval evaluation ready')

In [ ]:
# === EVALUATE BASE MODEL ===
print('='*60)
print('📊 Evaluating BASE model (chưa fine-tune)...')
print('='*60)
base_model, base_tok = load_model_for_eval(adapter_path=None)
base_score, base_results = run_humaneval(base_model, base_tok, 'Base')
del base_model  # Giải phóng VRAM
torch.cuda.empty_cache()

In [ ]:
# === EVALUATE FINE-TUNED MODEL ===
print('='*60)
print('📊 Evaluating FINE-TUNED model...')
print('='*60)
ft_model, ft_tok = load_model_for_eval(adapter_path=ADAPTER_PATH)
ft_score, ft_results = run_humaneval(ft_model, ft_tok, 'Fine-tuned')
del ft_model
torch.cuda.empty_cache()

## 📋 Cell 8: So sánh kết quả

In [ ]:
# Bảng so sánh
print('\n' + '='*60)
print('📋 KẾT QUẢ SO SÁNH — HumanEval pass@1')
print('='*60)
print(f'{"Model":<30} {"Score":<15} {"Passed/Total"}')
print('-'*60)
print(f'{"Qwen2.5-Coder-3B (Base)":<30} {base_score:<15.1f} {sum(base_results)}/{len(base_results)}')
print(f'{"Fine-tuned (OpenCodeInstruct)":<30} {ft_score:<15.1f} {sum(ft_results)}/{len(ft_results)}')
print('-'*60)
diff = ft_score - base_score
emoji = '✅ CẢI THIỆN' if diff > 0 else ('⚠️ GIẢM' if diff < 0 else '➡️ BẰNG')
print(f'Chênh lệch: {diff:+.1f}% → {emoji}')
print('='*60)

# Biểu đồ so sánh
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
bars = ax.bar(['Base Model', 'Fine-tuned'], [base_score, ft_score],
              color=['#3498db', '#e74c3c'], width=0.5)
for bar, score in zip(bars, [base_score, ft_score]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{score:.1f}%', ha='center', fontsize=14, fontweight='bold')
ax.set_ylabel('pass@1 (%)', fontsize=12)
ax.set_title('HumanEval: Base vs Fine-tuned', fontsize=14)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('humaneval_comparison.png', dpi=100)
plt.show()
print('📊 Biểu đồ đã lưu → humaneval_comparison.png')

## 🧪 Cell 9: Test thử model fine-tuned

In [ ]:
# Load lại fine-tuned model để test
ft_model, ft_tok = load_model_for_eval(adapter_path=ADAPTER_PATH)

test_prompts = [
    'Tạo API POST /users với FastAPI, dùng service layer và repository pattern',
    'Fix lỗi: connection pool exhausted trong SQLAlchemy',
    'Viết Dockerfile tối ưu cho FastAPI app',
]

for i, prompt in enumerate(test_prompts, 1):
    print(f'\n{"="*60}')
    print(f'🧪 Test {i}: {prompt}')
    print('='*60)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text = ft_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = ft_tok(text, return_tensors='pt').to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=512, do_sample=True,
                                temperature=0.7, top_p=0.9, pad_token_id=ft_tok.eos_token_id)
    response = ft_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(response)

## 📥 Cell 10: Download model về máy

In [ ]:
# Nén adapter để download
!cd output/qwen-senior-dev && zip -r /content/qwen_senior_dev_adapter.zip final_adapter/
print('\n📥 Download file: qwen_senior_dev_adapter.zip')
print('   Cách dùng trên máy local:')
print('   1. Giải nén vào thư mục project')
print('   2. Chạy: python src/inference.py')

from google.colab import files
files.download('/content/qwen_senior_dev_adapter.zip')